In [9]:
import os, subprocess
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

MODEL = os.environ.get("MODEL_ID")

In [10]:
TOOLS = [
    {
        "name": "bash",
        "description": "Run a shell command",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string"
                }
            },
            "required": ["command"]
        }
    }
]

In [11]:
query1 = "How many .md files are in the current directory?"

In [12]:
messages = [
    {
        "role": "user",
        "content": query1
    }
]

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'}]

In [13]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

### Tasks

#### 1. Find the tool_use block. Read its id and input["command"]

In [14]:
print(response.content[0].type)
print(response.content[0].id)
print(response.content[0].name)
print(response.content[0].input["command"])
print(response.content)

tool_use
toolu_019wA8D26LHrEQbs2nHN8PAm
bash
ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -1a | grep -c '\.md$'
[ToolUseBlock(id='toolu_019wA8D26LHrEQbs2nHN8PAm', caller=DirectCaller(type='direct'), input={'command': "ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -1a | grep -c '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)]


In [15]:
len(response.content)

1

#### 2. Run that command in a terminal and copy the output.

#### 3. Append the model's turn to messages with the assistant role, then append a user message containing one tool_result. It needs the matching tool_use_id and the output as content.

In [16]:
messages.append(
    {
        "role": "assistant",
        "content": response.content
    }
)

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_019wA8D26LHrEQbs2nHN8PAm', caller=DirectCaller(type='direct'), input={'command': "ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -1a | grep -c '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)]}]

In [17]:
command = response.content[0].input["command"]
command

"ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -1a | grep -c '\\.md$'"

In [18]:
result = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120).stdout
result

'       1\n1\n'

In [19]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[0].id,
                "content": result
            }
        ]
    }
)

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_019wA8D26LHrEQbs2nHN8PAm', caller=DirectCaller(type='direct'), input={'command': "ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -1a | grep -c '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_019wA8D26LHrEQbs2nHN8PAm',
    'content': '       1\n1\n'}]}]

#### 4. Call client.messages.create(...) again and look at the new stop_reason.

In [20]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

In [21]:
response

Message(id='msg_011ChU68RA16JDvsvTr2SEe1', container=None, content=[ToolUseBlock(id='toolu_01VER2dAsYeFpqF4Ap3dkb39', caller=DirectCaller(type='direct'), input={'command': "ls -la | grep '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)], diagnostics=None, model='claude-opus-5-5', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=471, output_tokens=58, output_tokens_details=OutputTokensDetails(thinking_tokens=0), server_tool_use=None, service_tier='standard', speed='standard'))

In [22]:
print(response.content[0])
print(response.content[1])

ToolUseBlock(id='toolu_01VER2dAsYeFpqF4Ap3dkb39', caller=DirectCaller(type='direct'), input={'command': "ls -la | grep '\\.md$'"}, name='bash', type='tool_use', toolset_name=None)


IndexError: list index out of range

In [ ]:
print(response.stop_reason)
print(response.content[1].text)


In [ ]:
query2 = "Whats the capital of France?"

In [ ]:
messages = [
    {
        "role": "user",
        "content": query2
    }
]

messages

In [ ]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

In [ ]:
print(response.content[0].type)
print(response.content)

In [ ]:
print(response.stop_reason)
print(response.content[0].text)

In [23]:
lista = [{'role': 'user', 'content': 'whats the current folder?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01MCqWgcH1q5nKp9EwfcckPP', caller=DirectCaller(type='direct'), input={'command': 'pwd'}, name='bash', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01MCqWgcH1q5nKp9EwfcckPP', 'content': '/Users/carlos/Projects/learn-claude-code/my_harness'}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="You're in `/Users/carlos/Projects/learn-claude-code/my_harness`.", type='text')]}]

NameError: name 'ToolUseBlock' is not defined